# 🔎 Lab 04: BERT 中文命名實體識別 (NER)

**使用 `bert-base-chinese` 做 BIO 序列標註**

---

### 🎯 學習目標
- 處理 BIO 標註格式
- 對齊 WordPiece subword 標籤
- 使用 `seqeval` 計算 entity-level F1
- 提取實體進行互動測試

### 📊 數據集
**MSRA 中文 NER** (PER/LOC/ORG 三類) 或倉庫自帶離線示例

### 🛠 技術棧
Transformers, PyTorch, seqeval

### ⏱ 預計時間
- GPU: **必須** (Colab T4 足夠)
- 運行時間: ~3 分鐘

### 🚀 GPU 加速
**菜單 → 更改執行階段類型 → 硬件加速器 → GPU**

---

## 1. 安裝

In [ ]:
!pip install -q transformers datasets accelerate seqeval
import torch
print(f'✅ PyTorch {torch.__version__}')
print(f'✅ CUDA: {torch.cuda.is_available()}')

## 2. 載入 BIO 數據

In [ ]:
import os
import urllib.request
from pathlib import Path

TAG2ID = {
    'O': 0, 'B-PER': 1, 'I-PER': 2,
    'B-LOC': 3, 'I-LOC': 4,
    'B-ORG': 5, 'I-ORG': 6,
    'B-MISC': 7, 'I-MISC': 8,
}
ID2TAG = {v: k for k, v in TAG2ID.items()}

def read_bio(path):
    sents, tags = [], []
    tokens, tag_seq = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sents.append(tokens); tags.append(tag_seq)
                    tokens, tag_seq = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                tokens.append(parts[0])
                tag_seq.append(parts[-1])
    if tokens:
        sents.append(tokens); tags.append(tag_seq)
    return sents, tags

DATA_DIR = '/content/data' if os.path.exists('/content') else './data'
if not os.path.exists(f'{DATA_DIR}/sample_train.txt'):
    os.makedirs(DATA_DIR, exist_ok=True)
    !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/04-ner-with-bert/data/sample_train.txt -O {DATA_DIR}/sample_train.txt
    !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/04-ner-with-bert/data/sample_test.txt -O {DATA_DIR}/sample_test.txt

train_sents, train_tags = read_bio(f'{DATA_DIR}/sample_train.txt')
test_sents, test_tags = read_bio(f'{DATA_DIR}/sample_test.txt')
print(f'📊 訓練: {len(train_sents)} 句, 測試: {len(test_sents)} 句')
print(f'示例: {list(zip(train_sents[0], train_tags[0]))}')

## 3. 載入 BERT

In [ ]:
MODEL_NAME = 'bert-base-chinese'
MAX_LEN = 128
EPOCHS = 5
BATCH_SIZE = 8
LR = 3e-5

from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(TAG2ID), id2label=ID2TAG, label2id=TAG2ID,
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'🧠 設備: {device}, 參數量: {sum(p.numel() for p in model.parameters()):,}')

## 4. 構建 Dataset (subword 對齊)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class NERDataset(Dataset):
    def __init__(self, sents, tags, tokenizer, tag2id, max_len=128):
        self.sents, self.tags, self.tokenizer, self.tag2id, self.max_len = sents, tags, tokenizer, tag2id, max_len
    def __len__(self): return len(self.sents)
    def __getitem__(self, idx):
        words = self.sents[idx]
        labels = self.tags[idx]
        enc = self.tokenizer(words, is_split_into_words=True, truncation=True,
                              padding='max_length', max_length=self.max_len, return_tensors='pt')
        word_ids = enc.word_ids(0)
        label_ids = []
        prev = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev:
                label_ids.append(self.tag2id.get(labels[wid], 0))
            else:
                label_ids.append(-100)
            prev = wid
        return {
            'input_ids': enc['input_ids'][0],
            'attention_mask': enc['attention_mask'][0],
            'labels': torch.tensor(label_ids, dtype=torch.long),
        }

train_loader = DataLoader(NERDataset(train_sents, train_tags, tokenizer, TAG2ID, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(NERDataset(test_sents, test_tags, tokenizer, TAG2ID, MAX_LEN),
                         batch_size=BATCH_SIZE*2)
print(f'✅ Train: {len(train_loader)} batches, Test: {len(test_loader)} batches')

## 5. 訓練

In [ ]:
import torch.nn as nn
from seqeval.metrics import f1_score as seq_f1, classification_report as seq_report

total_steps = len(train_loader) * EPOCHS
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device); am = batch['attention_mask'].to(device); lab = batch['labels'].to(device)
        out = model(input_ids=ids, attention_mask=am)
        pred = out.logits.argmax(-1).cpu().tolist()
        label = lab.cpu().tolist()
        for p_seq, l_seq in zip(pred, label):
            p_tags = [ID2TAG[p] for p, l in zip(p_seq, l_seq) if l != -100]
            t_tags = [ID2TAG[l] for l in l_seq if l != -100]
            all_preds.append(p_tags); all_labels.append(t_tags)
    return all_preds, all_labels

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids = batch['input_ids'].to(device); am = batch['attention_mask'].to(device); lab = batch['labels'].to(device)
        out = model(input_ids=ids, attention_mask=am, labels=lab)
        out.loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
    pred_tags, true_tags = evaluate(test_loader)
    f1 = seq_f1(true_tags, pred_tags)
    print(f'Epoch {epoch}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | F1={f1:.4f}')

## 6. 最終評估

In [ ]:
pred_tags, true_tags = evaluate(test_loader)
print(seq_report(true_tags, pred_tags))

## 7. 互動測試

In [ ]:
def extract_entities(text):
    words = list(text)
    enc = tokenizer(words, is_split_into_words=True, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(device)
    model.eval()
    with torch.no_grad():
        out = model(**enc)
        pred_ids = out.logits.argmax(-1)[0].cpu().tolist()
    word_ids = enc.word_ids(0)
    entities = []
    prev = None
    current = []
    for w, wid, pid in zip(words, word_ids, pred_ids):
        if wid is not None and wid != prev:
            tag = ID2TAG[pid]
            if tag.startswith('B-'):
                if current:
                    entities.append(current)
                current = [(w, tag[2:])]
            elif tag.startswith('I-') and current:
                current.append((w, tag[2:]))
            else:
                if current:
                    entities.append(current)
                    current = []
        prev = wid
    if current: entities.append(current)
    return entities

for t in [
    '張忠謀在台積電工作',
    '黃仁勳創辦了 NVIDIA 公司',
    '馬斯克投資了特斯拉和 SpaceX',
    '蘋果公司在加州',
]:
    ents = extract_entities(t)
    res = ', '.join([''.join([w for w, _ in e]) + f'({e[0][1]})' for e in ents])
    print(f'{t:30s} → {res if res else "(無實體)"}')

## 🎉 恭喜完成 Lab 04！

**你學到了**：
- ✅ BIO 標註格式處理
- ✅ Subword 標籤對齊
- ✅ Entity-level F1 評估
- ✅ 中文 NER 完整流程

### 🚀 下一步

👉 [Lab 05: RAG 問答系統](../05-rag-qa-system/lab.ipynb) — 檢索增強生成

### 📚 延伸閱讀

- [BERT for NER](https://huggingface.co/docs/transformers/tasks/token_classification)
- [中文 NER 數據集](https://github.com/SophonPlus/ChineseNlpCorpus)
- [CRF 層改進](https://arxiv.org/abs/1603.01360)